In [1]:
import sys
sys.path.append('..')
import cv2
import json
import torch
import torch.nn as nn
import numpy as np
import torch.nn.functional as F
from cying.nn import VisionModel
from PIL import Image
from tqdm import tqdm
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from cityscapesscripts.helpers.labels import labels
from torch_linear_assignment import batch_linear_assignment
from torchinfo import summary

label_to_Id = {label.name: label.trainId for label in labels}

class CityscapesDataset(Dataset):
    def __init__(self, root, split, target_size, num_query):
        self.root = Path(root)
        self.split = split
        self.size = target_size
        self.num_query = num_query

        gt_dir = self.root / 'gtFine' / split
        self.json_files = sorted(gt_dir.glob('*/*_gtFine_polygons.json'))

    def __len__(self):
        return len(self.json_files)

    def __getitem__(self, idx):
        json_path = self.json_files[idx]
        with open(json_path, 'r') as f:
            data = json.load(f)

        img_path = Path(str(json_path).replace('_gtFine_polygons.json', '_leftImg8bit.png').replace('gtFine', 'leftImg8bit'))
        image = np.array(Image.open(img_path).convert('RGB'))  

        h, w = image.shape[:2]
        target_h, target_w = self.size
        scale = min(target_h / h, target_w / w)
        new_h, new_w = int(h * scale), int(w * scale)

        image_resized = torch.tensor(cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LINEAR), dtype=torch.float32).permute(2, 0, 1) / 255.0
        image = torch.tile(
            image_resized,
            (
                1,
                (target_h + new_h - 1) // new_h, 
                (target_w + new_w - 1) // new_w   
            ),
        )[:, :target_h, :target_w]
        padding_mask = torch.ones((target_h, target_w), dtype=bool)
        padding_mask[:new_h, :new_w] = False

        masks = torch.zeros((self.num_query, target_h, target_w), dtype=torch.uint8)
        labels = torch.full((self.num_query,), -1, dtype=torch.long)
        bboxes = torch.zeros((self.num_query, 4), dtype=torch.float32)

        valid_idx = 0
        for obj in data['objects']:
            label_str = obj['label']
            label_id = label_to_Id.get(label_str, -1)
            if label_id == -1 or label_id == 255:
                continue

            polygon = np.array(obj['polygon'], dtype=np.int32)
            x_coords = polygon[:, 0]
            y_coords = polygon[:, 1]
            x_min, x_max = x_coords.min(), x_coords.max()
            y_min, y_max = y_coords.min(), y_coords.max()

            polygon_scaled = (polygon * scale).astype(np.int32)
            bbox_scaled = [x_min * scale/target_w, y_min * scale/target_h, x_max * scale/target_w, y_max * scale/target_h]

            mask = np.zeros((target_h, target_w), dtype=np.uint8)
            cv2.fillPoly(mask, [polygon_scaled], 1)

            masks[valid_idx] = torch.as_tensor(mask, dtype=torch.uint8)
            labels[valid_idx] = torch.as_tensor(label_id, dtype=torch.long)
            bboxes[valid_idx] = torch.as_tensor(bbox_scaled, dtype=torch.float32)

            valid_idx += 1

        return image, labels, bboxes, masks, padding_mask

In [2]:
class SetCriterion(nn.Module):
    def __init__(self, num_classes, loss_weights, matcher_weights, eov_coef=0.1):
        super().__init__()
        self.num_classes = num_classes
        self.loss_weights = loss_weights
        self.matcher_weights = matcher_weights
        self.eov_coef = eov_coef

        empty_weight = torch.ones(num_classes + 1)
        empty_weight[-1] = eov_coef
        self.register_buffer("empty_weight", empty_weight)
    
    def generalized_box_iou(self, boxes1, boxes2):
        area1 = (boxes1[..., 2] - boxes1[..., 0]) * \
                (boxes1[..., 3] - boxes1[..., 1])
        area2 = (boxes2[..., 2] - boxes2[..., 0]) * \
                (boxes2[..., 3] - boxes2[..., 1])

        lt = torch.max(boxes1[:, :, None, :2], boxes2[:, None, :, :2])
        rb = torch.min(boxes1[:, :, None, 2:], boxes2[:, None, :, 2:])
        wh = (rb - lt).clamp(min=0)
        inter = wh[..., 0] * wh[..., 1]

        union = area1[:, :, None] + area2[:, None, :] - inter

        iou = inter / union.clamp(min=1e-6)

        lt_c = torch.min(boxes1[:, :, None, :2], boxes2[:, None, :, :2])
        rb_c = torch.max(boxes1[:, :, None, 2:], boxes2[:, None, :, 2:])
        wh_c = (rb_c - lt_c).clamp(min=0)
        area_c = wh_c[..., 0] * wh_c[..., 1]

        giou = iou - (area_c - union) / area_c.clamp(min=1e-6)
        return giou

    @torch.no_grad()
    def hungarian_matcher(self, outputs, targets):
        pred_logits = outputs["pred_logits"]
        pred_boxes = outputs["pred_boxes"]
        tgt_labels = targets["labels"]
        tgt_boxes = targets["boxes"]

        invalid_tgt = tgt_labels < 0

        pred_prob = pred_logits.softmax(-1)
        cost_class = - pred_prob.gather(
            2, tgt_labels[:,None,:].clamp(min=0).expand(-1, pred_logits.size(1), -1)
        )
        cost_bbox = torch.cdist(pred_boxes, tgt_boxes, p=1)
        cost_giou = - self.generalized_box_iou(pred_boxes, tgt_boxes)

        C = self.matcher_weights['class'] * cost_class \
            + self.matcher_weights['bbox'] * cost_bbox \
            + self.matcher_weights['giou'] * cost_giou

        C = C.masked_fill(invalid_tgt[:,None,:], 1e6)

        assignment = batch_linear_assignment(C)

        assignment = assignment.masked_fill(
            invalid_tgt.gather(1, assignment), -1
        )
        return assignment

    def forward(self, outputs, targets):
        tgt_idx = self.hungarian_matcher(outputs, targets)
        
        valid = tgt_idx >= 0

        pred_logits = outputs["pred_logits"]
        pred_boxes = outputs["pred_boxes"]
        pred_masks = outputs["pred_masks"]

        tgt_labels_pad = targets["labels"]
        tgt_boxes_pad = targets["boxes"]
        tgt_masks_pad = targets["masks"]
        padding_mask = targets["padding_mask"]

        H_t, W_t = tgt_masks_pad.shape[-2:]

        safe_idx = tgt_idx.clamp(min=0)
        matched_labels = tgt_labels_pad.gather(1, safe_idx)
        matched_boxes = tgt_boxes_pad.gather(
            1, safe_idx[..., None].expand(-1, -1, 4)
        )
        matched_masks = tgt_masks_pad.gather(
            1, safe_idx[:, :, None, None].expand(-1, -1, H_t, W_t)
        )

        target_classes = matched_labels.masked_fill(~valid, self.num_classes)

        loss_ce = F.cross_entropy(
            pred_logits.transpose(1, 2),
            target_classes,
            weight=self.empty_weight,
        )

        num_matched = valid.sum().clamp(min=1.0)

        l1_per = F.l1_loss(pred_boxes, matched_boxes, reduction="none").sum(-1)
        giou = self.generalized_box_iou(pred_boxes, matched_boxes).diagonal(dim1=-2, dim2=-1)
        loss_bbox = (l1_per * valid).sum() / num_matched
        loss_giou = ((1.0 - giou) * valid).sum() / num_matched

        valid_pixel = (~padding_mask).unsqueeze(1)
        pred_m = pred_masks * valid_pixel
        tgt_m  = matched_masks * valid_pixel

        inter = (pred_m * tgt_m).sum((-1, -2))
        union = pred_m.sum((-1, -2)) + tgt_m.sum((-1, -2))
        dice  = 1.0 - (2.0 * inter + 1.0) / (union + 1.0)
        loss_dice = (dice * valid).sum() / num_matched

        n_pix = valid_pixel.sum((-1, -2)).clamp(min=1.0)
        bce = F.binary_cross_entropy(
            pred_masks, matched_masks, reduction="none"
        ) * valid_pixel
        loss_bce = (bce.sum((-1, -2)) / n_pix[:, None] * valid).sum() / num_matched

        loss_dict = {
            "loss_ce": loss_ce,
            "loss_bbox": loss_bbox,
            "loss_giou": loss_giou,
            "loss_dice": loss_dice,
            "loss_bce": loss_bce,
        }
        loss_dict["total_loss"] = sum(
            self.loss_weights.get(k, 1.0) * v for k, v in loss_dict.items()
        )
        return loss_dict

In [3]:
cityscapes_root = "/root/autodl-tmp/cityscapes"
target_size = (256, 512)
batch_size = 8
num_query = 291
num_classes = 19
num_heads = 8
decoder_layers = 3
hidden_width = 256

num_epochs = 50
lr = 1e-3
num_workers = 4

device = torch.device("cuda:0")

train_dataset = CityscapesDataset(
    root = cityscapes_root,
    split = "train",
    target_size = target_size,
    num_query = num_query
)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True
)

valid_dataset = CityscapesDataset(
    root = cityscapes_root,
    split = "valid",
    target_size = target_size,
    num_query = num_query
)

valid_dataloader = DataLoader(
    valid_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers
)

test_dataset = CityscapesDataset(
    root = cityscapes_root,
    split = "test",
    target_size = target_size,
    num_query = num_query
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers
)

backbone_params = [
    {
        'size': target_size,
        'in_channels': 3,
        'out_channels': 8,
        'hidden_width': hidden_width,
        'spe_opt_size': (11, 11),
        'spa_opt_size': 11
    },
    {
        'size': target_size,
        'in_channels': 8,
        'out_channels': 16,
        'hidden_width': hidden_width,
        'spe_opt_size': (11, 11),
        'spa_opt_size': 11
    }
]

encoder_params = [
    {
        'size': (64, 128),
        'in_channels': 16,
        'out_channels': 32,
        'hidden_width': hidden_width,
        'spe_opt_size': (11, 11),
        'spa_opt_size': 11
    },
    {
        'size': (16, 32),
        'in_channels': 32,
        'out_channels': 64,
        'hidden_width': hidden_width,
        'spe_opt_size': (9, 9),
        'spa_opt_size': 7
    },
    {
        'size': (8, 16),
        'in_channels': 64,
        'out_channels': 128,
        'hidden_width': hidden_width,
        'spe_opt_size': (5, 5),
        'spa_opt_size': 5
    }
]

In [4]:
model = VisionModel(
    backbone_params = backbone_params,
    encoder_params = encoder_params,
    num_classes = num_classes,
    num_heads = num_heads,
    decoder_hidden_width = hidden_width,
    decoder_layers = decoder_layers,
    num_query = num_query
).to(device)

summary(model)

Layer (type:depth-idx)                        Param #
VisionModel                                   --
├─OperatorModel2d: 1-1                        --
│    └─Sequential: 2-1                        --
│    │    └─OperatorLayer2d: 3-1              4,212
│    │    └─OperatorLayer2d: 3-2              9,488
├─OperatorModel2d: 1-2                        --
│    └─Sequential: 2-2                        --
│    │    └─OperatorLayer2d: 3-3              18,976
│    │    └─OperatorLayer2d: 3-4              33,856
│    │    └─OperatorLayer2d: 3-5              62,848
├─VisionDecoder: 1-3                          74,496
│    └─ModuleList: 2-3                        --
│    │    └─VisionDecoderLayer: 3-6           199,936
│    │    └─VisionDecoderLayer: 3-7           199,936
│    │    └─VisionDecoderLayer: 3-8           199,936
├─VisionPredictionHeads: 1-4                  --
│    └─Linear: 2-4                            2,580
│    └─Sequential: 2-5                        --
│    │    └─Linear: 3-9 

In [ ]:
loss_weight = {
    "loss_ce":    1.0,
    "loss_bbox":  5.0,
    "loss_giou":  2.0,
    "loss_dice":  5.0,
    "loss_bce":   2.0,
}

matcher_weight = {
    "class": 1.0,
    "bbox": 5.0,
    "giou": 2.0
}

loss_fn = SetCriterion(
    num_classes=num_classes,
    loss_weights=loss_weight,
    matcher_weights=matcher_weight,
    eov_coef=0.1,
).to(device)

optim = torch.optim.Adam(model.parameters(), lr=lr)

for epoch in range(num_epochs):
    model.train()

    pbar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")

    for images, labels, bboxes, masks, padding_mask in pbar:
        optim.zero_grad(set_to_none=True)

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        bboxes = bboxes.to(device, non_blocking=True)
        padding_mask = padding_mask.to(device, non_blocking=True)

        targets = {
            "masks": masks.float(),
            "labels": labels,
            "boxes": bboxes,
            "padding_mask": padding_mask,
        }

        logits, boxes, masks = model(images, padding_mask)

        outputs = {
            "pred_logits": logits,
            "pred_boxes": torch.cat([boxes[..., :2], boxes[..., :2] + boxes[..., 2:]], dim=-1),
            "pred_masks": masks
        }

        losses = loss_fn(outputs, targets)

        losses["total_loss"].backward()

        optim.step()

        pbar.set_postfix({"loss": f"{losses['total_loss'].item():.4f}"})

Epoch 1/50:  48%|████▊     | 178/372 [00:45<00:46,  4.15it/s, loss=14.8660]